# Single camera calibration

In [1]:
pwd()

'/home/user/Documents/repos/pyptv/pyptv'

In [2]:
import optv
import pathlib
import os
import copy
from skimage.util import img_as_ubyte
from skimage.color import rgb2gray


import matplotlib.pyplot as plt
import mpl_interactions.ipyplot as iplt
%matplotlib tk


In [3]:
from optv.imgcoord import image_coordinates  # type: ignore
from optv.transforms import convert_arr_metric_to_pixel  # type: ignore
from optv.orientation import match_detection_to_ref  # type: ignore
from optv.orientation import external_calibration, full_calibration  # type: ignore
from optv.calibration import Calibration  # type: ignore
from optv.tracking_framebuf import TargetArray  # type: ignore

import parameters as par


In [4]:
# PyPTV works only inside the working folder
# working_folder = pathlib.Path('../tests/test_cavity/')
working_folder = pathlib.Path('/home/user/Downloads/test_8/')
# working_folder = pathlib.Path('c:/Users/Alex/Downloads/test_8')
os.chdir(working_folder)


working_folder = pathlib.Path('.')
par_path = working_folder / "parameters"

In [5]:
with open(par_path / "ptv.par", "r", encoding="utf-8") as f:
    n_cams = int(f.readline())
    
print(f'{n_cams} cameras setup')

4 cameras setup


In [6]:
from ptv import *

In [7]:
# Read all the parameters in respective structures
cpar, spar, vpar, track_par, tpar, cals, epar = py_start_proc_c(n_cams)

# we need for calibration:
tpar.read(b"parameters/detect_plate.par")


In [8]:
tpar.get_grey_thresholds()

array([120, 125, 125, 125], dtype=int32)

In [9]:
calParams = par.CalOriParams(n_cams, par_path)
calParams.read()

In [10]:
# Let's focus on one camera, i_cam
i_cam = 0

In [11]:
# read calibration image

imname = calParams.img_cal_name[i_cam]
im = imread(imname)
if im.ndim > 2:
    im = rgb2gray(im)

cal_image = img_as_ubyte(im)

In [12]:
# local directory to keep some data
camera = {'man_ori':np.zeros((4,)), '_x':[], '_y':[]}

In [13]:
# Loading manual parameters here
man_ori_path =  par_path / "man_ori.par"
with open(man_ori_path, "r", encoding='utf8') as f:
    for j in range(4):
        camera["man_ori"][j] = int(f.readline().strip())

In [14]:
camera

{'man_ori': array([ 1.,  8., 40., 17.]), '_x': [], '_y': []}

In [15]:
cal = Calibration()
tmp = cpar.get_cal_img_base_name(i_cam)
cal.from_file(tmp + b".ori", tmp + b".addpar")
    
print(cal.get_pos(), '\n', cal.get_angles())

[-57.28759858  70.74876204 285.98923369] 
 [-0.27337737 -0.20845443 -0.02966963]


In [16]:
fig, ax = plt.subplots()
ax.imshow(cal_image, cmap='gray')

In [17]:
# highpass
# high_passed = py_pre_processing_c(cal_images, cpar)
high_passed = preprocess_image(cal_image, 0, cpar, 25)

fig, ax = plt.subplots()
ax.imshow(high_passed, cmap='gray')

In [18]:
# detection
def mydetection(high_passed=high_passed, tpar=tpar, i_cam=i_cam, cpar=cpar):
    """ Returns np.array of x,y"""
    targs = target_recognition(high_passed, tpar, i_cam, cpar)
    return np.array([row.pos() for row in targs])

In [19]:
# detect all dots depending on tpar, cpar
xy = mydetection()

In [20]:
fig, ax = plt.subplots()
ax.imshow(high_passed)
ax.scatter(xy[:,0], xy[:,1], 35, marker='x', color='r')

In [21]:
# interactive single camera detection cell
# change parameters by sliders
# high-pass the image
# apply detection
# visualize it

In [22]:
def detection(threshold=tpar.get_grey_thresholds()[i_cam], 
              minpix = tpar.get_pixel_count_bounds()[0], 
              maxpix = tpar.get_pixel_count_bounds()[1],
             ):

    # adjust the threshold of a desired camera
    thresholds = tpar.get_grey_thresholds()
    thresholds[i_cam] = threshold
    tpar.set_grey_thresholds(thresholds)
            
    tpar.set_pixel_count_bounds((minpix, maxpix))
  
    # apply detection
    return mydetection(tpar=tpar)

In [23]:
fig, ax = plt.subplots()
# from matplotlib.widgets import Slider
# plt.subplots_adjust(bottom=0.25)
ax.imshow(high_passed, cmap='gray')
# axfreq = plt.axes([0.25, 0.1, 0.65, 0.03])
# slider = Slider(axfreq, label="freq", valmin=0.05, valmax=10)
# controls = iplt.plot(x, f, freq=slider, ax=ax)

controls = iplt.scatter(detection, 
                        ax=ax, 
                        threshold=(100,190,tpar.get_grey_thresholds()[i_cam]),
                        minpix=(25,150,tpar.get_pixel_count_bounds()[0]), 
                        maxpix=(160,700,tpar.get_pixel_count_bounds()[1]), 
                        parametric=True,
                        force_ipywidgets=True,
                       )
# plt.show()


In [24]:
controls.params
# next todo: write these to the respective parameter files

{'threshold': 100.0, 'minpix': 25.0, 'maxpix': 160.0}

In [25]:
tpar.get_grey_thresholds(), tpar.get_pixel_count_bounds()

(array([100, 125, 125, 125], dtype=int32), (25, 160))

In [26]:
detect_parameters = par.DetectPlateParams()

In [27]:
detect_parameters.read()

In [28]:
detect_parameters.filepath()

PosixPath('parameters/detect_plate.par')

In [29]:
tmp = detect_parameters.trait_get()
tmp
# detect_parameters.set()

{'path': PosixPath('parameters'),
 'exp_path': PosixPath('.'),
 'gvth_1': 120,
 'gvth_2': 125,
 'gvth_3': 125,
 'gvth_4': 125,
 'tol_dis': 500,
 'min_npix': 25,
 'max_npix': 500,
 'min_npix_x': 5,
 'max_npix_x': 100,
 'min_npix_y': 5,
 'max_npix_y': 100,
 'sum_grey': 500,
 'size_cross': 10}

In [30]:
tmp

{'path': PosixPath('parameters'),
 'exp_path': PosixPath('.'),
 'gvth_1': 120,
 'gvth_2': 125,
 'gvth_3': 125,
 'gvth_4': 125,
 'tol_dis': 500,
 'min_npix': 25,
 'max_npix': 500,
 'min_npix_x': 5,
 'max_npix_x': 100,
 'min_npix_y': 5,
 'max_npix_y': 100,
 'sum_grey': 500,
 'size_cross': 10}

In [31]:
detect_parameters.trait_set(list(tmp.values())[2:])

In [32]:
detect_parameters.trait_get()

{'path': PosixPath('parameters'),
 'exp_path': PosixPath('.'),
 'gvth_1': [120, 125, 125, 125, 500, 25, 500, 5, 100, 5, 100, 500, 10],
 'gvth_2': traits.trait_types.Int,
 'gvth_3': traits.trait_types.Int,
 'gvth_4': traits.trait_types.Int,
 'tol_dis': traits.trait_types.Int,
 'min_npix': traits.trait_types.Int,
 'max_npix': traits.trait_types.Int,
 'min_npix_x': traits.trait_types.Int,
 'max_npix_x': traits.trait_types.Int,
 'min_npix_y': traits.trait_types.Int,
 'max_npix_y': traits.trait_types.Int,
 'sum_grey': traits.trait_types.Int,
 'size_cross': traits.trait_types.Int}

In [33]:
detect_parameters.read()

In [34]:
detect_parameters.trait_set(tmp)
detect_parameters.write()

Opened file 



TypeError: int() argument must be a string, a bytes-like object or a number, not 'dict'

In [ ]:
# manual orientation points from parameters and calibration
# we have to read all four sets, then pick one


man_ori_path = "man_ori.dat"
with open(man_ori_path, "r") as f:
    for i in range(n_cams):
        if i == i_cam:
            camera["_x"], camera["_y"] = [], []
            for j in range(4):  # 4 orientation points
                line = f.readline().split()
                # print(line)
                # print(float(line[0]), float(line[1]))
                camera["_x"].append(float(line[0]))
                camera["_y"].append(float(line[1]))
    else:
        for j in range(4):  # 4 orientation points, skip
            line = f.readline().split()
        

fig, ax = plt.subplots()        
ax.imshow(high_passed, cmap='gray')
ax.scatter(camera["_x"], camera["_y"], 45, marker='+',color='red')
for x,y,s in zip(camera['_x'], camera['_y'], camera['man_ori']):
    ax.text(x,y,str(int(s)),color='r',fontsize=12)

In [ ]:
# later use for manual clicks
# https://matplotlib.org/stable/gallery/event_handling/cursor_demo.html
# skip this for now

In [ ]:
# read calibration block file, x,y,z known points

def _read_cal_points():
    return np.atleast_1d(
        np.loadtxt(
            pathlib.Path(calParams.fixp_name), # calblock.txt
            dtype=[("id", "i4"), ("pos", "3f8")],
            skiprows=0,
        ))

cal_points = _read_cal_points()

In [ ]:
def _project_cal_points(i_cam=i_cam):
    x, y = [], []
    for row in cal_points:
        projected = image_coordinates(
            np.atleast_2d(row["pos"]),
            cals[i_cam],
            cpar.get_multimedia_params(),
        )
        pos = convert_arr_metric_to_pixel(projected, cpar)

        x.append(pos[0][0])
        y.append(pos[0][1])
        
    return x,y

In [ ]:
# initial guess plot

fig, ax = plt.subplots()
ax.imshow(high_passed, cmap='gray')
x,y = _project_cal_points()
ax.scatter(x, y, 35, marker='+',color='y')

In [ ]:
# sortgrid
"""
Uses sortgrid function of liboptv to match between the
calibration points in the fixp target file and the targets
detected in the images
"""
sorted_targs = []
for i_cam in range(n_cams):
    fig, ax = plt.subplots(figsize=(6,6))

    targs = match_detection_to_ref(
        cals[i_cam],
        cal_points["pos"],
        detections[i_cam],
        cpar,
        101
    )
    x, y, pnr = [], [], []
    for t in targs:
        if t.pnr() != -999:
            pnr.append(cal_points["id"][t.pnr()])
            x.append(t.pos()[0])
            y.append(t.pos()[1])

    ax.imshow(high_passed[i_cam])
    ax.scatter(x,y,marker='.', color='r')
    for xx,yy,ss in zip(x,y,pnr):
        ax.text(xx,yy,ss,color='r')
    # ax.text(x,y,pnr)
    
    sorted_targs.append(targs)

In [ ]:
# full calibration

"""
fine tuning of ORI and ADDPAR

"""
scale = 5000

op = par.OrientParams()
op.read()

# recognized names for the flags:
names = [
    "cc",
    "xh",
    "yh",
    "k1",
    "k2",
    "k3",
    "p1",
    "p2",
    "scale",
    "shear",
]
op_names = [
    op.cc,
    op.xh,
    op.yh,
    op.k1,
    op.k2,
    op.k3,
    op.p1,
    op.p2,
    op.scale,
    op.shear,
]

flags = []
for name, op_name in zip(names, op_names):
    if op_name == 1:
        flags.append(name)

print(flags)

In [ ]:
sum_residuals = []
for num_iterations in range(10):
    # for num_iterations in range(1000):
    for i_cam in range(n_cams):  # iterate over all cameras

        targs = sorted_targs[i_cam]

        # print('Before fine tuning')
        # print(cals[i_cam].get_pos())
        # print(cals[i_cam].get_angles())

        pos = cals[i_cam].get_pos()
        ang = cals[i_cam].get_angles()

        residuals, targ_ix, err_est = full_calibration(
            cals[i_cam],
            cal_points["pos"],
            targs,
            cpar,
            flags,
        )

        sum_residuals.append(np.sum(residuals))

In [ ]:
# from pbi 
def fitness(solution, calib_targs, calib_detect, cpar):
    """
    Checks the fitness of an evolutionary solution of calibration values to 
    target points. Fitness is the sum of squares of the distance from each 
    guessed point to the closest neighbor.
    
    Arguments:
    solution - array, concatenated: position of intersection with Z=0 plane; 3 
        angles of exterior calibration; primary point (xh,yh,cc); 3 radial
        distortion parameters; 2 decentering parameters.
    calib_targs - a (p,3) array of p known points on the calibration target.
    calib_detect - a (d,2) array of d detected points in the calibration 
        target.
    cpar - a ControlParams object with image data.
    """
    # Breakdown of of agregate solution vector:
    inters = np.zeros(3)
    inters[:2] = solution[:2]
    R = solution[2]
    angs = solution[3:6]
    prim_point = solution[6:9]
    rad_dist = solution[9:12]
    decent = solution[12:14]
    
    # Compare known points' projections to detections:
    cal = gen_calib(inters, R, angs, glass_vec, prim_point, rad_dist,
                    decent)
    known_proj = image_coordinates(calib_targs, cal,
                                    cpar.get_multimedia_params())
    known_2d = convert_arr_metric_to_pixel(known_proj, cpar)
    dists = np.linalg.norm(known_2d[None, :, :] - calib_detect[:, None, :],
                            axis=2).min(axis=0)

    return np.sum(dists**2)

In [ ]:
def get_pos(inters, R, angs):
    # Transpose of http://planning.cs.uiuc.edu/node102.html
    # Also consider the angles are reversed when moving from camera frame to
    # global frame.
    s = np.sin(angs)
    c = np.cos(angs)
    pos = inters + R*np.r_[ s[1], -c[1]*s[0], c[1]*c[0] ]
    return pos

In [ ]:
def get_polar_rep(pos, angs):
    """
    Returns the point of intersection with zero Z plane, and distance from it.
    """
    s = np.sin(angs)
    c = np.cos(angs)
    zdir = -np.r_[ s[1], -c[1]*s[0], c[1]*c[0] ]
    
    c = -pos[2]/zdir[2]
    inters = pos + c*zdir
    R = np.linalg.norm(inters - pos)
    
    return inters[:2], R

In [ ]:
def gen_calib(inters, R, angs, glass_vec, prim_point, radial_dist, decent):
    pos = get_pos(inters, R, angs)
    return Calibration(pos, angs, prim_point, radial_dist, decent, 
        np.r_[1, 0], glass_vec)

In [ ]:
    
def parse_calib(
    cal: Calibration
    )-> np.ndarray:
    """parse calibration to solution

    Args:
        cal (Calibration): _description_

    Returns:
        np.ndarray: solution array
    """
    
    # Inverse it: 
    
    # Breakdown of of agregate solution vector:
    # inters = np.zeros(3)
    # inters[:2] = solution[:2]
    # R = solution[2]
    # angs = solution[3:6]
    # prim_point = solution[6:9]
    # rad_dist = solution[9:12]
    # decent = solution[12:14]
    
    inters = cal.get_pos() # including R
    angs = cal.get_angles()
    primary_point = cal.get_primary_point()
    rad_dist = cal.get_radial_distortion()
    decent = cal.get_decentering()
    
    return np.r_[inters, angs, primary_point, rad_dist, decent]

In [ ]:
def copy_calibration(cal):
    return Calibration(cal.get_pos(), 
                      cal.get_angles(), 
                      cal.get_primary_point(), 
                      cal.get_radial_distortion(), 
                      cal.get_decentering(), 
                      cal.get_affine(), 
                      cal.get_glass_vec())
    

In [ ]:
def project_cal_points(cal_points, calibration, cpar):
    x, y = [], []
    for row in cal_points:
        projected = image_coordinates(
            np.atleast_2d(row["pos"]),
            calibration,
            cpar.get_multimedia_params(),
        )
        pos = convert_arr_metric_to_pixel(projected, cpar)

        x.append(pos[0][0])
        y.append(pos[0][1])
        
    return x,y

In [ ]:
def get_targs(calibration, cal_points, detections, cpar):
    """gets matching targets in x,y """
    targs = match_detection_to_ref(
        calibration,
        cal_points["pos"],
        detections,
        cpar,
    )
    x, y, pnr = [], [], []
    for t in targs:
        if t.pnr() != -999:
            pnr.append(cal_points["id"][t.pnr()])
            x.append(t.pos()[0])
            y.append(t.pos()[1])
    return x,y


In [ ]:
def full_calibration(
                    cal: Calibration,
                    calib_detect: np.ndarray,
                    calib_targs: np.ndarray,
                    cpar: ControlParams,
                    flags,
                )-> Calibration:
    """ New full calibration using scipy minimize

    Returns:
        _type_: _description_
    """
    
    # define partial function to keep the arguments
    solution = parse_calib(cal)
    res = minimize(fitness, 
                            solution,
                            args = (
                                calib_targs, 
                                calib_detect, 
                                cpar)
    )
    
    res_cal = gen_calib(res)

    # res is the cal object that minimizes the fitness
    err_est = fitness(res_cal, calib_targs, calib_detect, cpar)
    

    return res_cal, err_est

In [ ]:
# for i_cam in range(1):
i_cam = 0
fig, ax = plt.subplots(figsize=(12,12))


tmp = copy_calibration(cals[i_cam])
solution = parse_calib(tmp)
print(f'solution is {solution}')

x,y = get_targs(tmp, cal_points, detections[i_cam], cpar)


ax.imshow(high_passed[i_cam],cmap='gray')
ax.scatter(x,y,marker='.', color='r')
for xx,yy,ss in zip(x,y,pnr):
    ax.text(xx,yy,ss,color='r')

xc,yc = project_cal_points(cal_points, tmp, cpar)
ax.scatter(xc, yc, 30, marker='+',color='g')
known_2d = np.c_[xc,yc]
calib_detect = np.c_[x,y]

dists = np.linalg.norm(known_2d[None, :, :] - calib_detect[:, None, :],
                        axis=2).min(axis=0)


print(f'error is {np.sum(dists**2)}')
    
    # return np.sum(dists**2)
    

In [ ]:
# for i_cam in range(1):
i_cam = 0
fig, ax = plt.subplots(figsize=(12,12))


tmp = copy_calibration(cals[i_cam])

# change one parameter at a time

eps = 0.01
# print(tmp.get_radial_distortion())


tmp.set_radial_distortion(tmp.get_radial_distortion() + np.array([eps, 0, 0]))
# print(tmp.get_radial_distortion())
# solution = parse_calib(tmp)
print(f'solution is {parse_calib(tmp)}')


x,y = get_targs(tmp, cal_points, detections[i_cam], cpar)


ax.imshow(high_passed[i_cam],cmap='gray')
ax.scatter(x,y,marker='.', color='r')
for xx,yy,ss in zip(x,y,pnr):
    ax.text(xx,yy,ss,color='r')

xc,yc = project_cal_points(cal_points, tmp, cpar)
ax.scatter(xc, yc, 30, marker='+',color='g')
known_2d = np.c_[xc,yc]
calib_detect = np.c_[x,y]

dists = np.linalg.norm(known_2d[None, :, :] - calib_detect[:, None, :],
                        axis=2).min(axis=0)


print(f'error is {np.sum(dists**2)}')

In [ ]:
# Explanation of the Calibration structure

# from calibration.pyx and calibration.pxd in optv:
# pos = exterior.x0,y0,z0
# angles = exterior. omega, phi, kappa
# primary point = xh, yh, cc
# glass = x,y,z
# radial_distortion = k1,k2,k3, works in r = x**2 + y**2, p = k1 r**2 + k2 r**4 + k3 r**6
# decentering = p1, p2
# affine = scale, shear

In [ ]:
i_cam = 0
# , ax = plt.subplots(figsize=(12,12))
tmp = copy_calibration(cals[i_cam])

def fitness(eps):
   
    rd = tmp.get_radial_distortion()
    rd[2] = eps
    tmp.set_radial_distortion(rd)

    x,y = get_targs(tmp, cal_points, detections[i_cam], cpar)
    xc,yc = project_cal_points(cal_points, tmp, cpar)
    
    known_2d = np.c_[xc,yc]
    calib_detect = np.c_[x,y]

    dists = np.linalg.norm(known_2d[None, :, :] - calib_detect[:, None, :],
                            axis=2).min(axis=0)
    
    return np.sum(dists**2)

In [ ]:
from scipy.optimize import minimize
sol = minimize(fitness, 0. , method='Nelder-Mead',options ={'disp':True, 'xatol': 1e-6})

In [ ]:
sol.x

In [ ]:
# for i_cam in range(1):

fig, ax = plt.subplots(figsize=(12,12))

tmp = copy_calibration(cals[i_cam])

rd = tmp.get_radial_distortion()
print(f' Initial radial distortion {rd}')
rd[1] = sol.x
tmp.set_radial_distortion(rd)

solution = parse_calib(tmp)
print(f'solution is {solution}')

x,y = get_targs(tmp, cal_points, detections[i_cam], cpar)


ax.imshow(high_passed[i_cam],cmap='gray')
ax.scatter(x,y,marker='.', color='r')
for xx,yy,ss in zip(x,y,pnr):
    ax.text(xx,yy,ss,color='r')

xc,yc = project_cal_points(cal_points, tmp, cpar)
ax.scatter(xc, yc, 30, marker='+',color='g')
known_2d = np.c_[xc,yc]
calib_detect = np.c_[x,y]

dists = np.linalg.norm(known_2d[None, :, :] - calib_detect[:, None, :],
                        axis=2).min(axis=0)


print(f'error is {np.sum(dists**2)}')

In [ ]:
# Now it's about time to run the complete calibration again, with k1 modified and pos() and angles(), to get the 
# new sortgrid and then the new calibration again? 
#

In [ ]:
tmp = copy_calibration(cals[i_cam])

def project_points(k1=0, k2=0, k3=0):
    """Project points based on a scalar parameter """

    rd = tmp.get_radial_distortion()
    rd[0] = k1
    rd[1] = k2
    rd[2] = k3
    tmp.set_radial_distortion(rd)

    #solution = parse_calib(tmp)
    # print(f'solution is {solution}')

    x,y = get_targs(tmp, cal_points, detections[i_cam], cpar)
    
    return np.c_[x,y]

fig, ax = plt.subplots()
ax.imshow(high_passed[i_cam], cmap='gray')

ax.imshow(high_passed[i_cam],cmap='gray')
# ax.scatter(x,y,marker='.', color='r')
# for xx,yy,ss in zip(x,y,pnr):
#     ax.text(xx,yy,ss,color='r')
    
xc,yc = project_cal_points(cal_points, tmp, cpar)
ax.scatter(xc, yc, 30, marker='+',color='g')

controls = iplt.scatter(project_points, 
                        ax=ax, 
                        k1=np.linspace(-1e-2,1e-2),
                        k2=np.linspace(-1e-2,1e-2),
                        k3=np.linspace(-1e-2,1e-2),
                        parametric=True,
                        marker='.',
                        color='r'
                       )
plt.show()